# Waveform susceptibility in anisotropic Womersley flow

This notebook installs the public package, executes the complete six-artery analysis, and stores tables, arrays, figures, and the analysis summary on Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import shutil
import subprocess

REPOSITORY_URL = 'https://github.com/khalid-saqr/picoNewton.git'
REPOSITORY_REF = os.environ.get('PICONEWTON_REF', 'main')
REPOSITORY_ROOT = Path('/content/picoNewton_waveform_susceptibility')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_ROOT = Path('/content/drive/MyDrive/picoNewton_waveform_susceptibility') / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
subprocess.run(['git', 'checkout', REPOSITORY_REF], cwd=REPOSITORY_ROOT, check=True)
print('Output directory:', OUTPUT_ROOT)

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPOSITORY_ROOT / 'picoNewton_v3')], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(REPOSITORY_ROOT / 'waveform_susceptibility')], check=True)

In [ ]:
command = [
    'piconewton-waveform-susceptibility',
    '--output', str(OUTPUT_ROOT),
    '--radial-order', '150',
    '--time-points', '2048',
    '--quadrature-nodes', '256',
    '--validation-epsilon', '0.08',
    '--figure-dpi', '300',
]
subprocess.run(command, check=True)

In [ ]:
import json
summary = json.loads((OUTPUT_ROOT / 'analysis_summary.json').read_text())
summary

In [ ]:
from IPython.display import Image, display
for figure in sorted((OUTPUT_ROOT / 'figures').glob('*.png')):
    display(Image(filename=str(figure), width=900))